# 01 · EDA on raw vote accounts
Loads the cached pull (`data/raw/vote_accounts.parquet`). Build it with `python -m src.extract` first; this notebook never queries BigQuery.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.data_io import read_raw
from src.features import dedup_latest, n_records

In [ ]:
raw = read_raw()
print("Shape:", raw.shape)
raw.info()

## Missingness

In [ ]:
missing = pd.DataFrame({
    "missing_count": raw.isna().sum(),
    "missing_pct": raw.isna().mean() * 100,
}).sort_values("missing_pct", ascending=False)
missing

## Deduplicate (latest snapshot per `node_pubkey`)

In [ ]:
v = dedup_latest(raw)
print(f"Rows: {len(raw):,} -> {len(v):,}  (dropped {len(raw) - len(v):,} duplicates)")

## Column distributions

In [ ]:
v["block_timestamp"] = pd.to_datetime(v["block_timestamp"])
v.groupby(v["block_timestamp"].dt.date).size().plot(figsize=(10, 4), title="Vote account records over time")
plt.show()

In [ ]:
print(v["commission"].describe())
v["commission"].hist(bins=30); plt.title("Commission"); plt.show()
v["commission"].value_counts().sort_index()

In [ ]:
print(v["lamports"].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
v["lamports"].hist(bins=50, ax=ax[0]); ax[0].set_title("Lamports")
np.log10(v["lamports"] + 1).hist(bins=50, ax=ax[1]); ax[1].set_title("log10(1 + lamports)")
plt.show()

In [ ]:
v["root_slot"].hist(bins=50); plt.title("Root slot"); plt.show()

## Nested columns

In [ ]:
counts = pd.DataFrame({c: v[c].map(n_records) for c in config.NESTED_COLUMNS})
display(counts.describe())
counts.hist(bins=30, figsize=(12, 6)); plt.tight_layout(); plt.show()

In [ ]:
for i in range(3):
    print(v.iloc[i]["node_pubkey"])
    for rec in (v.iloc[i]["epoch_credits"] or [])[:5]:
        print("  ", rec)

In [ ]:
def first(rec_list, key):
    return rec_list[0].get(key) if rec_list else None

v["latest_vote_slot"] = pd.to_numeric(v["last_timestamp"].map(lambda x: first(x, "slot")), errors="coerce")
v["latest_vote_timestamp"] = pd.to_datetime(v["last_timestamp"].map(lambda x: first(x, "timestamp")), errors="coerce", utc=True)
display(v[["latest_vote_slot", "latest_vote_timestamp"]].describe())
v["latest_vote_timestamp"].hist(bins=50); plt.title("Latest vote timestamp"); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
v["rent_epoch"].hist(bins=30, ax=ax[0]); ax[0].set_title("rent_epoch")
v["space"].hist(bins=30, ax=ax[1]); ax[1].set_title("space")
plt.show()